# Comparación de Q-Learning y DQN en MountainCar-v0

Este notebook entrena y compara los dos agentes del proyecto. Observaremos qué tan rápido mejoran, qué tan estable es cada entrenamiento y cómo se comportan al final.

Una recompensa de `-200` significa que el automóvil no llegó a la bandera en los 200 pasos disponibles. Una recompensa menos negativa es mejor.

## 1. Configuración

Los valores de episodios son los recomendados por el ejercicio. El DQN puede tardar varios minutos en CPU. Para una prueba rápida se pueden usar 200 episodios.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

from mountain_car.agents import DQNAgent, QLearningAgent

SEMILLA = 7
EPISODIOS_Q = 20_000
EPISODIOS_DQN = 2_500
EPISODIOS_EVALUACION = 10
random.seed(SEMILLA)
np.random.seed(SEMILLA)
print('Configuración lista')

: 

## 2. Funciones para evaluar y resumir

La evaluación usa siempre la acción más conveniente según el agente, sin exploración.

In [ ]:
def evaluar(agente, episodios=EPISODIOS_EVALUACION):
    entorno = gym.make('MountainCar-v0')
    recompensas, pasos, llego = [], [], []
    for episodio in range(episodios):
        estado, _ = entorno.reset(seed=SEMILLA + episodio)
        total = 0.0
        for paso in range(1, 201):
            accion, _ = agente.predict(estado, deterministic=True)
            estado, recompensa, terminado, truncado, _ = entorno.step(int(accion))
            total += recompensa
            if terminado or truncado:
                recompensas.append(total)
                pasos.append(paso)
                llego.append(bool(terminado))
                break
    entorno.close()
    return {'media': float(np.mean(recompensas)), 'desviacion': float(np.std(recompensas)), 'mejor': float(np.max(recompensas)), 'peor': float(np.min(recompensas)), 'pasos_medios': float(np.mean(pasos)), 'bandera': int(sum(llego)), 'recompensas': recompensas}

def media_movil(valores, ventana=100):
    if len(valores) < ventana:
        return np.asarray(valores)
    return np.convolve(valores, np.ones(ventana) / ventana, mode='valid')

def mostrar_resumen(nombre, resultado):
    print(f'\n{nombre}')
    print(f"  Recompensa media: {resultado['media']:.2f} +/- {resultado['desviacion']:.2f}")
    print(f"  Mejor / peor: {resultado['mejor']:.2f} / {resultado['peor']:.2f}")
    print(f"  Pasos medios: {resultado['pasos_medios']:.1f}")
    print(f"  Llegó a la bandera: {resultado['bandera']}/{len(resultado['recompensas'])}")

: 

## 3. Entrenamiento

Cada agente comienza desde cero y conserva la recompensa de cada episodio para hacer la gráfica.

In [ ]:
agente_q = QLearningAgent('MountainCar-v0')
historial_q = agente_q.train(total_episodes=EPISODIOS_Q, log_interval=1000)

agente_dqn = DQNAgent('MountainCar-v0')
historial_dqn = agente_dqn.train(total_episodes=EPISODIOS_DQN, log_interval=250)
print('Entrenamiento terminado')

## 4. Evidencia visual del entrenamiento

La línea suave permite ver la tendencia general, mientras que la línea tenue muestra la variación de cada episodio.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(historial_q, alpha=0.15, color='tab:blue', label='Q-Learning: episodios')
plt.plot(np.arange(99, len(historial_q)), media_movil(historial_q), color='tab:blue', label='Q-Learning: media móvil')
plt.plot(historial_dqn, alpha=0.15, color='tab:orange', label='DQN: episodios')
plt.plot(np.arange(99, len(historial_dqn)), media_movil(historial_dqn), color='tab:orange', label='DQN: media móvil')
plt.axhline(-200, color='gray', linestyle='--', linewidth=1, label='Sin llegar: -200')
plt.xlabel('Episodio'); plt.ylabel('Recompensa total')
plt.title('Evolución de la recompensa durante el entrenamiento')
plt.legend(); plt.grid(alpha=0.25); plt.show()

## 5. Evaluación final

La mejor recompensa es la más cercana a cero, porque cada paso resta un punto.

In [ ]:
resultado_q = evaluar(agente_q)
resultado_dqn = evaluar(agente_dqn)
mostrar_resumen('Q-Learning', resultado_q)
mostrar_resumen('DQN', resultado_dqn)

In [ ]:
nombres = ['Q-Learning', 'DQN']
medias = [resultado_q['media'], resultado_dqn['media']]
banderas = [resultado_q['bandera'], resultado_dqn['bandera']]
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].bar(nombres, medias, color=['tab:blue', 'tab:orange']); ejes[0].set_title('Recompensa media final'); ejes[0].set_ylabel('Recompensa'); ejes[0].axhline(-200, color='gray', linestyle='--'); ejes[0].grid(axis='y', alpha=0.25)
ejes[1].bar(nombres, banderas, color=['tab:blue', 'tab:orange']); ejes[1].set_title('Llegadas a la bandera'); ejes[1].set_ylabel(f'Episodios de {EPISODIOS_EVALUACION}'); ejes[1].set_ylim(0, EPISODIOS_EVALUACION); ejes[1].grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show()

## 6. Lectura sencilla

- Q-Learning usa una tabla: es más fácil de entender y suele ser estable en este entorno pequeño, pero depende de la discretización.
- DQN usa una red neuronal: puede aprender relaciones más flexibles, aunque necesita memoria, más ajustes y puede variar más.
- La comparación debe considerar la media, la variación y cuántas veces llegó a la bandera, no solo un episodio aislado.
- Los valores impresos y las gráficas de las celdas anteriores son la evidencia que debe incluirse en el informe final.